# 0 · Basics: Python for MIRtoolbox users

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansgomez87/py-mir-toolbox/blob/main/docs/chapters/00_basics.ipynb)

This chapter covers what you need from Python before computing any music feature: running a notebook, working with arrays, plotting, listening to sounds, loading a sound file with mirpy, and reading the numbers mirpy gives back. If you know MATLAB, most ideas carry over directly.

## Running a notebook

A notebook is a list of cells. Text cells, like this one, explain; code cells run Python. Click a code cell and press **Shift+Enter** to run it: its output appears below it, and the next cell is selected. It works like running a MATLAB script one section at a time.

Cells share their variables, so run them from top to bottom. If something goes wrong, restart and run everything again: in Colab, *Runtime → Restart session and run all*; in JupyterLab, *Kernel → Restart Kernel and Run All Cells*.

## Setup

In Colab, the next cell installs the libraries used in the course. On your own computer, see *Getting started*.

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic(
        "pip", "install -q 'py-mir-toolbox==0.1.*' librosa essentia"
    )

## Arrays: the Python version of MATLAB matrices

NumPy arrays play the role of MATLAB matrices. Two differences cause most mistakes:

1. indexing starts at 0, so `x[0]` is the first sample (MATLAB `x(1)`); and
2. slices exclude the end, so `x[0:3]` is three samples (MATLAB `x(1:3)`).

In [ ]:
import numpy as np

x = np.array([10, 20, 30, 40, 50])
print(x[0])  # first element, MATLAB x(1)
print(x[0:3])  # first three, MATLAB x(1:3)
print(x[-1])  # last element, MATLAB x(end)

## A sound is an array of samples

A digital sound is a sequence of amplitude values taken `sr` times per second (the sampling rate). Here we synthesize one second of a 440 Hz tone, the A above middle C.

In [ ]:
sr = 22050  # samples per second
t = np.arange(sr) / sr  # time of each sample, in seconds
tone = 0.5 * np.sin(2 * np.pi * 440 * t)
print(tone.shape, tone[:5])

## Listening and plotting

In a notebook, `IPython.display.Audio` gives you a player, and Matplotlib draws the waveform.

In [ ]:
from IPython.display import Audio

Audio(tone, rate=sr)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t[:200], tone[:200])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("First 200 samples of a 440 Hz tone")
plt.show()

## Loading a sound file

`mir.audio` is MIRtoolbox `miraudio`. Here we first save our tone as a WAV file, then read it back, as you would read your own recordings.

In [ ]:
import soundfile as sf

import mirpy as mir

sf.write("tone.wav", tone, sr)  # save the tone as a WAV file
a = mir.audio("tone.wav")  # MATLAB: a = miraudio('tone.wav');
print(a.sr)  # its sampling rate

For your own recordings, give the path of the file, for example `mir.audio("stimuli/piece01.wav")`. In Colab, first upload the files with the folder icon on the left. WAV, FLAC, OGG and MP3 files can be read.

## What is inside a mirpy result

Every mirpy result keeps its numbers in `.data`, a NumPy array with three axes, always in the same order: **(channels, elements, frames)**.

- **channels**: the audio channels (1 for a mono sound), or the bands of a filterbank;
- **elements**: the samples of a sound, the frequencies of a spectrum, or 1 for a feature with one value per frame, such as the RMS energy;
- **frames**: 1 for a whole sound; more once it is cut into frames (chapter 1).

MIRtoolbox orders them differently: (element, frame, channel).

In [ ]:
rms_whole = mir.rms(a)  # one value for the whole sound
rms_curve = mir.rms(a, frame=True)  # one value per 50 ms frame

print(a.data.shape)  # (1 channel, 22050 samples, 1 frame)
print(rms_whole.data.shape)  # (1, 1, 1): a single number
print(rms_curve.data.shape)  # (1, 1, number of frames)

So `rms_curve.data[0, 0]` is the curve of the first channel, as a one-dimensional array: the form used in the plots of the next chapters. `a.data[0, :, 0]` gives the samples of a sound.

`mir.get_data` removes the axes of length 1, like MATLAB `mirgetdata`, and every result has a `.plot()` method:

In [ ]:
print(mir.get_data(rms_whole))  # a plain number
print(mir.get_data(rms_curve).shape)  # (number of frames,)

rms_curve.plot()
plt.show()

## Next

[1 · Frames and dynamics](01_dynamics.ipynb) cuts a sound into short frames, the first step of almost every MIRtoolbox feature, and measures its energy. You can already explore framing on the [interactive site](https://ccrma.stanford.edu/~jsgomezc/py-mir-toolbox/).